In [2]:
%pip install scikit-learn
%pip install pandas
%pip install joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
import joblib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
DATA_PATH = "../data/fake_reviews_dataset[1].csv"
MODEL_PATH = "../artifacts/review_detector_v2_calibrated.joblib"

In [6]:
import os
os.listdir(r"C:\Users\HP\Desktop\AI PROJECT")

['amazon_reviews.csv',
 'fake_reviews_dataset[1].csv',
 'Untitled4.ipynb',
 'yelp_reviews.csv']

In [7]:
def train_and_save(random_state=42):

    # Load dataset
    df = pd.read_csv(DATA_PATH)

    # Remove empty text rows
    df = df.dropna(subset=["text_"])

    # Split data
    X_train, X_val, y_train, y_val = train_test_split(
        df["text_"],
        df["label"],
        test_size=0.2,
        random_state=random_state,
        stratify=df["label"]
    )

    # Base classifier
    base_clf = LinearSVC(dual=False, class_weight="balanced")

    # Calibrated classifier
    calibrated = CalibratedClassifierCV(base_estimator=base_clf, cv=3)

    # Pipeline
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=20000)),
        ("clf", calibrated)
    ])

    # Train model
    pipeline.fit(X_train, y_train)

    # Save model
    joblib.dump(pipeline, MODEL_PATH)

    # Validation accuracy
    accuracy = pipeline.score(X_val, y_val)

    return accuracy


In [9]:

import joblib
from pathlib import Path

In [11]:
BASE_DIR = Path(os.getcwd())

In [12]:
MODEL_PATH = BASE_DIR / "artifacts" / "review_detector_v2_calibrated.joblib"


In [13]:
LABEL_MAP = {"CG": "AI-generated", "OR": "Human"}

print("Model path:", MODEL_PATH)
print("Label map:", LABEL_MAP)

Model path: C:\Users\HP\artifacts\review_detector_v2_calibrated.joblib
Label map: {'CG': 'AI-generated', 'OR': 'Human'}


In [15]:
def load_model():
    return joblib.load(MODEL_PATH)

def bucket_score(prob_fake: float):
    if prob_fake < 0.25:
        bucket = "Non-AI"
        explanation = "Model strongly favors human-authored language patterns."
    elif prob_fake < 0.55:
        bucket = "Maybe AI"
        explanation = "Signals are mixed; manual review recommended."
    elif prob_fake < 0.80:
        bucket = "Likely AI"
        explanation = "Linguistic markers align with automated text."
    else:
        bucket = "Surely AI"
        explanation = "Model is highly confident this review is synthetic."
    return bucket, explanation

def score_reviews(model, texts):
    import numpy as np
    probs = model.predict_proba(texts)
    fake_idx = list(model.classes_).index("CG")
    outputs = []
    for text, proba_vec in zip(texts, probs):
        prob_fake = float(proba_vec[fake_idx])
        bucket, explanation = bucket_score(prob_fake)
        outputs.append({
            "text": text,
            "prob_fake": prob_fake,
            "prob_human": 1 - prob_fake,
            "bucket": bucket,
            "explanation": explanation,
            "predicted_label": "CG" if prob_fake >= 0.5 else "OR"
        })
    return outputs

In [17]:
%pip install flask flask-cors
from flask import Flask, request, jsonify
from flask_cors import CORS
import os

Note: you may need to restart the kernel to use updated packages.


In [29]:
API_KEY = "12345"  # you can change this later
MODEL_PATH = "artifacts/review_detector_v2_calibrated.joblib"

In [33]:
def load_model():
    if not os.path.exists(MODEL_PATH):
        print("ERROR: Model file not found at:", MODEL_PATH)
        return None
    return joblib.load(MODEL_PATH)

In [34]:
model = None

In [35]:
if os.path.exists(MODEL_PATH):
    model = joblib.load(MODEL_PATH)
    print("Model loaded successfully")
else:
    print("Model file not found. API will run but predictions won't work.")



Model file not found. API will run but predictions won't work.


In [36]:
app = Flask(__name__)
CORS(app)

In [37]:
@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

In [38]:
def authenticate(req):
    return req.headers.get("X-API-Key") == API_KEY

In [39]:
@app.route("/predict", methods=["POST"])
def predict():

    if not authenticate(request):
        return jsonify({"error": "Unauthorized"}), 401

    if model is None:
        return jsonify({"error": "Model not loaded"}), 500

    data = request.json
    text = data.get("text", "")

    if text == "":
        return jsonify({"error": "No text provided"}), 400

    prediction = model.predict([text])[0]

    return jsonify({
        "review": text,
        "prediction": str(prediction)
    })

In [ ]:
app.run(port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


In [ ]:
MODEL_PATH = "review_detector_v2_calibrated.joblib"